<a href="https://colab.research.google.com/github/Prashant43226/CUDA-Programming/blob/main/RMSNorm-Profiling-across-CUDA-Triton-JAX/XLA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RMS Norm has been implemented in :

1.   CUDA
2.   Triton

And then performance is compared for both of them

It is observed from this that Custom CUDA Kernels outperform Triton Kernels for simple to large workloads

Observed Performance improvement:


1.   For small workloads like (4,256): 2x
2.   For moderate workloads like (16,4096):12.36x




In [5]:
%%writefile rmsnorm.cpp
#include <cuda.h>
#include <cuda_runtime.h>
#include <device_launch_parameters.h>
#include <stdio.h>
#include <stdlib.h>
#include <torch/extension.h>
#include <cooperative_groups.h>

namespace cg = cooperative_groups;

template<typename T>
__global__ void rmsnorm_forward_kernel(const T* x, const T* weight, T* output, float eps, int N){
  int row_idx = blockIdx.x;
  const T* row_x = x + row_idx * N;  //Each row in the input vector will be split into subrows
  //So essentially a tensor of size 4x500 will be split into 4 rows of 500 dimension
  T* row_output = output + row_idx * N;

  float sum_sq = 0.0f;
  //This will aggreate the input values in the first block of each row of lets say size 256(block size)
  for(int i = threadIdx.x; i < N; i += blockDim.x){
    float val = (float)row_x[i];
    sum_sq += val * val;
  }

  // Intra-warp reduction using cooperative groups
  cg::thread_block block = cg::this_thread_block();
  cg::thread_block_tile<32> tile = cg::tiled_partition<32>(block);

  // This will sum the values into each warp's first thread in the first block of each row.
  for(int offset = tile.size() / 2; offset > 0; offset >>= 1){
    sum_sq += tile.shfl_down(sum_sq, offset);
  }

  //Since registers cannot be shared across warps hence need shared memory
  // Shared memory to hold the sums from each warp (max 32 warps for 1024 threads)
  __shared__ float shared_sums[32];
  int warp_id = threadIdx.x / 32;
  int lane_id = threadIdx.x % 32;

  // Each warp's lane 0 writes its local warp sum to shared memory
  if(lane_id == 0){
    shared_sums[warp_id] = sum_sq;
  }
  block.sync();

  //This will sum across all the warp's first thread to a simgle register of first warp's first thread.
  // Inter-warp reduction using the first warp
  if(warp_id == 0){
    sum_sq = (threadIdx.x < (blockDim.x / 32)) ? shared_sums[threadIdx.x] : 0.0f;

    for(int offset = 16; offset > 0; offset >>= 1){
      sum_sq += __shfl_down_sync(0xFFFFFFFF, sum_sq, offset);
    }
  }

  // Re-use a single shared float to broadcast final result
  __shared__ float final_shared_sum;
  if(threadIdx.x == 0){
    final_shared_sum = 1.0f / sqrtf((sum_sq / N) + eps);
  }
  block.sync();

  float rsqrt_rms = final_shared_sum;

  // Compute final normalised output and scale by weight
  for(int i = threadIdx.x; i < N; i += blockDim.x){
    // Added multiplying by the weight[i] parameter as required by RMSNorm
    row_output[i] = (T)((float)row_x[i] * rsqrt_rms * (float)weight[i]);
  }
}

torch::Tensor rmsnorm_forward_cuda(torch::Tensor x, torch::Tensor weight, float eps){
  int N = x.size(-1);
  auto out = torch::empty_like(x);
  int num_token = x.numel() / N;

  // Keep threads to a clean multiple of warp size (256 threads = 8 warps)
  int threads = 256;
  int blocks = num_token;

  AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "rmsnorm_forward_cuda", ([&] {
    rmsnorm_forward_kernel<scalar_t><<<blocks, threads>>>(
      x.data_ptr<scalar_t>(),
      weight.data_ptr<scalar_t>(),
      out.data_ptr<scalar_t>(),
      eps,
      N
    );
  }));
  return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m){
  m.def("rmsnorm_forward", &rmsnorm_forward_cuda, "rmsnorm_forward");
}


Writing rmsnorm.cpp


In [6]:
!ls

rmsnorm.cpp  sample_data


In [9]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 11.1 MB/s eta 0:00:00


In [16]:
!rm -rf /root/.cache/torch_extensions/

In [22]:
import os
import torch
from torch.utils.cpp_extension import load

# 1. Manually set CUDA_HOME so PyTorch can locate nvcc compiler tools
# Colab always installs CUDA under /usr/local/cuda
os.environ["CUDA_HOME"] = "/usr/local/cuda"

# 2. Read the C++/CUDA code file
with open('/content/rmsnorm.cpp', 'r') as f:
    cuda_source = f.read()

# 3. Write out to an explicit .cu file so PyTorch triggers the CUDA builder
with open('/content/rmsnorm_ext.cu', 'w') as f:
    f.write(cuda_source)

# 4. Compile using the correct arguments
print("Compiling CUDA kernel... (this takes ~1-2 minutes)")
rmsnorm_cuda = load(
    name="rmsnorm_cuda",
    sources=['/content/rmsnorm_ext.cu'],
    extra_cflags=['-O4'], #increment this value in case you face cache issue(file not found issue)
    verbose=True
)

print("Compilation successful! Module loaded.")

Compiling CUDA kernel... (this takes ~1-2 minutes)
Compilation successful! Module loaded.


In [24]:
# Create test tensors on the GPU
device = 'cuda'
x = torch.randn(4, 256, device=device)       # 4 tokens, hidden dimension 256
weight = torch.ones(256, device=device)       # Weights matching hidden dimension
eps = 1e-5

# Call your custom compiled CUDA function
output = rmsnorm_cuda.rmsnorm_forward(x, weight, eps)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Output Sample (First row, first 5 elements):\n", output[0, :5])


Input shape: torch.Size([4, 256])
Output shape: torch.Size([4, 256])
Output Sample (First row, first 5 elements):
 tensor([-0.9864,  0.1911, -0.5937,  1.0147, -2.0876], device='cuda:0')


Same RMS NORM CUDA Kernel in Triton

In [1]:
!pip install triton

In [3]:
import torch
import triton
import triton.language as tl

@triton.jit
def rmsnorm_forward_kernel(x_ptr,wt_ptr,y_ptr,eps,N,BLOCK_SIZE:tl.constexpr):
  row=tl.program_id(0)
  cols=tl.arange(0,BLOCK_SIZE)
  mask=cols<N

  x_offset=row*N+cols;
  x=tl.load(x_ptr+x_offset,mask=mask,other=0.0)

  variance=tl.sum(x*x,axis=0)/N
  rstd=tl.rsqrt(variance+eps)

  w=tl.load(wt_ptr+cols,mask=mask,other=0.0)
  y=x*rstd*w
  tl.store(y_ptr+x_offset,y,mask=mask)

def rmsnorm_triton(x,weights,eps=1e-5):
  assert x.is_cuda
  assert weights.is_cuda

  x=x.contiguous()
  weights=weights.contiguous()

  N=x.shape[-1]

  output=torch.empty_like(x)

  rows=x.numel()//N

  BLOCK_SIZE=triton.next_power_of_2(rows)
  rmsnorm_forward_kernel[BLOCK_SIZE,N](x,weights,output,eps,N,BLOCK_SIZE=BLOCK_SIZE)

  return output


device='cuda'
x=torch.randn(4,256,device=device)
weight=torch.ones(256,device=device)
eps=1e-5

output=rmsnorm_triton(x,weight,eps)
print("Input:")
print(x)
print("\n Weights: ")
print(weight)
print("\n Output: ")
print(output)


Input:
tensor([[-1.2800, -1.5197, -0.0301,  ...,  0.2294, -1.2126, -0.5223],
        [-0.1900,  1.4496, -0.0986,  ...,  0.8665, -0.3538,  0.4307],
        [ 0.3424,  0.0133, -0.1163,  ...,  1.1544, -0.9455, -0.9135],
        [-0.6938, -1.1030,  0.3272,  ..., -0.9912, -0.6009,  1.3206]],
       device='cuda:0')

 Weights: 
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 

In [25]:
%whos

Variable                 Type             Data/Info
---------------------------------------------------
batch                    int              8
benchmark                function         <function benchmark at 0x7e150fc20180>
cuda_source              str              #include <cuda.h>\n#inclu<...> "rmsnorm_forward");\n}\n
device                   str              cuda
eps                      float            1e-05
f                        TextIOWrapper    <_io.TextIOWrapper name='<...>ode='w' encoding='utf-8'>
hidden                   int              4096
load                     function         <function load at 0x7e13aa725620>
os                       module           <module 'os' (frozen)>
output                   Tensor           tensor([[-0.9864,  0.1911<...>\n       device='cuda:0')
rmsnorm_cuda             module           <module 'rmsnorm_cuda_v1'<...>cuda/rmsnorm_cuda_v1.so'>
rmsnorm_forward_kernel   JITFunction      JITFunction(__main__:rmsnorm_forward_kernel)
rmsnorm_t

Add a Benchmarking Function to Compare Both

In [30]:
import torch
def benchmark(fn,*args,warmup=50,repeat=200):
  for _ in range(warmup):
    fn(*args)
  torch.cuda.synchronize()

  start=torch.cuda.Event(enable_timing=True)
  end=torch.cuda.Event(enable_timing=True)
  start.record()
  for _ in range(repeat):
    fn(*args)
  end.record()
  torch.cuda.synchronize()
  elapsed_ms=start.elapsed_time(end)
  return elapsed_ms/repeat


x=torch.randn(16,4096,device='cuda')
weight=torch.ones(4096,device='cuda')

eps=1e-5

triton_time=benchmark(rmsnorm_triton,x,weight,eps)
print(f"Triton: {triton_time:.2f} ms")

cuda_time=benchmark(rmsnorm_cuda.rmsnorm_forward,x,weight,eps)
print(f"CUDA: {cuda_time:.2f} ms")

print(f"Speedup: {triton_time/cuda_time:.2f}x")

Triton: 0.22 ms
CUDA: 0.02 ms
Speedup: 12.36x
